In [1]:
%pip install python-dotenv requests

  Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
  Using cached requests-2.34.0-py3-none-any.whl (73 kB)
  Using cached charset_normalizer-3.4.7-cp311-cp311-win_amd64.whl (159 kB)
  Using cached idna-3.14-py3-none-any.whl (72 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl (135 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import os
import requests
from dotenv import load_dotenv

# Force reload to ensure fresh .env keys
load_dotenv(override=True) 

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

# Ensure this URL matches your key type (Test vs Production)
TOKEN_URL = "https://www.fuel-finder.service.gov.uk/api/v1/oauth/generate_access_token"

def get_access_token():
    # Credentials must be sent as form data
    payload = {
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET
    }
    
    # Standard OAuth 2.0 headers
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    # Use 'data=' for form-encoding
    response = requests.post(TOKEN_URL, data=payload, headers=headers, timeout=10)
    
    if response.status_code == 401:
        print("401 ERROR: Your keys are being rejected. Are you using TEST keys on a LIVE URL?")
        
    response.raise_for_status()
    return response.json().get("access_token")
if __name__ == "__main__":
    try:
        print("Authenticating...")
        token = get_access_token()
        print("Fetching all UK fuel prices (this may take a moment)...")
        
        data = get_fuel_prices(token)
        
        # The API returns a 'stations' list or similar object
        print(f"Success! Data received for stations.")
        # Print the first station's data as a sample
        if isinstance(data, list) and len(data) > 0:
            print(data[0])
        else:
            print(data)
            
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

Authenticating...
Fetching all UK fuel prices (this may take a moment)...
Forbidden: Check if your API key has 'Information Recipient' permissions.
An error occurred: 403 Client Error: Forbidden for url: https://www.fuel-finder.service.gov.uk/api/v1/pfs/fuel-prices?batch-number=1
